# 📈 S&P 500 Companies — Exploratory Data Analysis

> **Dataset:** S&P 500 constituent companies (503 records, 7 features)  
> **Goal:** Uncover sector trends, geographic concentration, historical patterns, and index evolution

---

## Table of Contents
1. [Setup & Data Loading](#1-setup--data-loading)
2. [Data Cleaning & Feature Engineering](#2-data-cleaning--feature-engineering)
3. [Exploratory Analysis](#3-exploratory-analysis)
   - [Sector Distribution](#31-sector-distribution)
   - [Index Additions Over Time](#32-index-additions-over-time)
   - [Geographic Analysis](#33-geographic-analysis)
   - [Founding Decade Trends](#34-founding-decade-trends)
   - [Sector × Sub-Industry Heatmap](#35-sector--sub-industry-heatmap)
4. [Key Business Insights](#4-key-business-insights)
5. [Conclusions & Next Steps](#5-conclusions--next-steps)

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Blues_d')
%matplotlib inline

df = pd.read_csv('../data/sp500_companies.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
# Check for missing values
df.isnull().sum()

## 2. Data Cleaning & Feature Engineering

In [ ]:
# Parse dates and numerics
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['founded']    = pd.to_numeric(df['founded'], errors='coerce')

# Derived features
df['year_added']     = df['date_added'].dt.year
df['decade_added']   = (df['year_added'] // 10 * 10).astype('Int64')
df['founded_decade'] = (df['founded']    // 10 * 10).astype('Int64')
df['hq_state']       = df['headquarters'].str.split(',').str[-1].str.strip()

print('Feature engineering complete.')
df[['year_added','decade_added','founded_decade','hq_state']].head()

## 3. Exploratory Analysis

### 3.1 Sector Distribution

In [ ]:
sector_counts = df['sector'].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(sector_counts.index, sector_counts.values,
               color=sns.color_palette('Blues_d', len(sector_counts)))
for bar in bars:
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(int(bar.get_width())), va='center', fontweight='bold')
ax.set_title('S&P 500 Companies by Sector', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Companies')
ax.set_xlim(0, sector_counts.max() + 15)
plt.tight_layout()
plt.show()

print('\nSector breakdown:')
print(sector_counts.sort_values(ascending=False).to_string())

### 3.2 Index Additions Over Time

In [ ]:
decade_counts = (
    df.dropna(subset=['decade_added'])
    .groupby(df['decade_added'].astype(int)).size()
    .reset_index(name='count')
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=decade_counts, x='decade_added', y='count', palette='Blues_d', ax=ax)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width()/2, p.get_height() + 0.5),
                ha='center', fontweight='bold')
ax.set_title('Companies Added to S&P 500 — by Decade', fontsize=14, fontweight='bold')
ax.set_xlabel('Decade')
ax.set_ylabel('Companies Added')
plt.tight_layout()
plt.show()

### 3.3 Geographic Analysis

In [ ]:
state_counts = df['hq_state'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=state_counts.values, y=state_counts.index, palette='Blues_d', ax=ax)
for p in ax.patches:
    ax.text(p.get_width() + 0.3, p.get_y() + p.get_height()/2,
            str(int(p.get_width())), va='center', fontweight='bold')
ax.set_title('Top 10 States by S&P 500 HQ Count', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Companies')
ax.set_xlim(0, state_counts.max() + 10)
plt.tight_layout()
plt.show()

ny = state_counts.get('New York', 0)
ca = state_counts.get('California', 0)
tx = state_counts.get('Texas', 0)
print(f'NY + CA + TX: {ny+ca+tx} companies = {(ny+ca+tx)/len(df)*100:.1f}% of the index')

### 3.4 Founding Decade Trends

In [ ]:
pivot = (
    df.dropna(subset=['founded_decade'])
    .assign(founded_decade=lambda x: x['founded_decade'].astype(int))
    .query('founded_decade >= 1900')
    .groupby(['founded_decade','sector']).size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(14, 6))
pivot.plot(kind='bar', stacked=True, colormap='tab20', ax=ax, width=0.8)
ax.set_title('S&P 500 Companies Founded per Decade (by Sector, 1900+)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Founding Decade')
ax.set_ylabel('Number of Companies')
ax.legend(loc='upper left', fontsize=7, ncol=2)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 3.5 Sector × Sub-Industry Heatmap

In [ ]:
top_sectors = df['sector'].value_counts().head(6).index
sub = df[df['sector'].isin(top_sectors)]
top_sub = sub['sub_industry'].value_counts().head(15).index
pivot_heat = sub.groupby(['sector','sub_industry']).size().unstack(fill_value=0)
pivot_heat = pivot_heat[[c for c in top_sub if c in pivot_heat.columns]]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot_heat, cmap='Blues', linewidths=0.4, linecolor='white',
            annot=True, fmt='d', ax=ax, cbar_kws={'label': 'Companies'})
ax.set_title('Sector × Sub-Industry Matrix (Top 6 Sectors, Top 15 Sub-Industries)',
             fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

## 4. Key Business Insights

In [ ]:
insights = {
    'Dominant Sector': f"Industrials leads with 79 companies (15.7% of index)",
    'Geographic Concentration': f"NY, CA & TX host 34.4% of S&P 500 headquarters",
    'Oldest Member': f"BNY Mellon (est. 1784) — 241+ years of continuous operation",
    'Peak Index Addition Decade': f"2010s saw 141 additions — highest in any decade",
    'Tech Surge': f"Information Technology has 73 companies, 3rd-largest sector",
    'Post-Pandemic Reshuffling': f"91 companies joined since 2020 — active rebalancing",
}

for i, (key, val) in enumerate(insights.items(), 1):
    print(f'{i}. {key}\n   → {val}\n')

## 5. Conclusions & Next Steps

### What we found
- **Industrials dominate** the index by company count, while tech companies have grown significantly
- **Geographic clustering** around NY, CA, TX reflects both financial and tech industry hubs
- **Index composition is dynamic** — 91 additions since 2020 alone, showing active rebalancing
- **Oldest companies are financial** — Financials sector has the most century-old institutions

### Potential extensions
- 📡 **Merge with financial data** (e.g. Yahoo Finance via `yfinance`) to add market cap, P/E, EPS
- 📊 **Sector performance analysis** — compare returns across sectors over time
- 🗺️ **Choropleth map** — visualize HQ density by US state using `plotly` or `folium`
- 🤖 **Predictive modeling** — can sector + founding year predict index tenure?
- 📉 **Survivorship bias study** — analyze companies that were removed from the index